In [45]:
import pandas as pd

df_sv = pd.read_csv("Data/sale_varieties.csv")


def norm_null(s):
    t = s.astype(str).str.strip().str.upper()
    return t.replace({"NULL": pd.NA, "NAN": pd.NA, "NONE": pd.NA, "": pd.NA})


df_sv["variety_id"] = pd.to_numeric(df_sv["id"], errors="coerce").astype("Int64")
_src = norm_null(df_sv["source_variety_id"])
df_sv["source_variety_id"] = pd.to_numeric(_src, errors="coerce").astype("Int64")

_idx = df_sv.set_index("variety_id")["source_variety_id"]
parent_direct = {}
for v in df_sv["variety_id"].dropna().astype(int):
    v = int(v)
    p = _idx.loc[v] if v in _idx.index else pd.NA
    parent_direct[v] = v if pd.isna(p) else int(p)

children_map = {}
for _, row in df_sv.dropna(subset=["variety_id"]).iterrows():
    v, p = int(row["variety_id"]), row["source_variety_id"]
    if not pd.isna(p):
        children_map.setdefault(int(p), []).append(v)


def get_direct_parent(variety_id):
    return parent_direct.get(int(variety_id), pd.NA)

def get_descendants(source_variety_id):
    if pd.isna(source_variety_id):
        return []
    rid = int(source_variety_id)
    return sorted([rid] + children_map.get(rid, []))

#print("descendants of 294 :", get_descendants(294))
#print("child 869 direct_parent:", get_direct_parent(751))
#descendants of 294 : [294, 387, 454, 499, 596, 706, 751]
#child 869 direct_parent: 294

In [46]:
df_orders = pd.read_csv("Data/order_scaler_products.csv")

df_orders.head()

,variety_id,grade,amount,order_date,order_time
0,140,out,105.0,14020216,11:31
1,140,out,105.0,14011228,12:42
2,140,out,105.0,14010719,16:33
3,140,out,104.0,14020329,11:24
4,140,out,104.0,14010816,12:13


In [47]:
# تبدیل ساعت به عدد
df_orders["time_num"] = (
    df_orders["order_time"]
    .str.replace(":", "", regex=False)
    .astype(int)
)

# تابع تبدیل به بازه
def map_time_range(x):
    if 600 <= x <= 859:
        return "06:00-08:59"
    elif 900 <= x <= 1159:
        return "09:00-11:59"
    elif 1200 <= x <= 1459:
        return "12:00-14:59"
    elif 1500 <= x <= 1659:
        return "15:00-17:59"
    elif 1700 <= x <= 2059:
        return "18:00-20:59"
    elif 2100 <= x <= 2359:
        return "21:00-23:59"
    else:
        return "other"

# جایگزینی ساعت با بازه
df_orders["order_time"] = df_orders["time_num"].apply(map_time_range)

# حذف ستون کمکی
df_orders.drop(columns=["time_num"], inplace=True)

df_orders.head()

,variety_id,grade,amount,order_date,order_time
0,140,out,105.0,14020216,09:00-11:59
1,140,out,105.0,14011228,12:00-14:59
2,140,out,105.0,14010719,15:00-17:59
3,140,out,104.0,14020329,09:00-11:59
4,140,out,104.0,14010816,12:00-14:59


In [48]:
# گروه‌بندی و جمع amount
df_grouped = (
    df_orders
    .groupby(
        ["variety_id", "grade", "order_date", "order_time"],
        as_index=False
    )["amount"]
    .sum()
)


df_grouped["source_id"] = df_grouped["variety_id"].apply(get_direct_parent)
df_grouped = df_grouped.dropna(subset=["source_id"]).reset_index(drop=True)
df_grouped["descendants"] = (
    df_grouped["source_id"]
    .apply(get_descendants)
    .apply(lambda x: "-".join(map(str, x)) if isinstance(x, list) else "")
)

In [49]:
time_order = [
    "06:00-08:59",
    "09:00-11:59",
    "12:00-14:59",
    "15:00-17:59",
    "18:00-20:59",
    "21:00-23:59"
]

df_grouped["order_time"] = pd.Categorical(
    df_grouped["order_time"],
    categories=time_order,
    ordered=True
)

# مرتب‌سازی
df_grouped = df_grouped.sort_values(
    ["order_date", "variety_id", "grade", "order_time"]
)

# محاسبه inventory کاهشی
def calc_inventory(x):
    total = x.sum()
    return total - x.cumsum() + x

df_grouped["inventory"] = (
    df_grouped
    .groupby(["order_date", "variety_id", "grade"])["amount"]
    .transform(calc_inventory)
)

df_grouped.head()

,variety_id,grade,order_date,order_time,amount,source_id,descendants,inventory
38508,56,1,14010315,18:00-20:59,1.51,56,56,1.51
68946,121,1,14010315,18:00-20:59,6.00,121,121,6.00
71086,126,1,14010315,18:00-20:59,13.00,126,126,52.00
71087,126,1,14010315,21:00-23:59,39.00,126,126,39.00
0,6,1,14010316,18:00-20:59,1.00,6,6,1.00


In [50]:
df_grouped.head(5)

,variety_id,grade,order_date,order_time,amount,source_id,descendants,inventory
38508,56,1,14010315,18:00-20:59,1.51,56,56,1.51
68946,121,1,14010315,18:00-20:59,6.00,121,121,6.00
71086,126,1,14010315,18:00-20:59,13.00,126,126,52.00
71087,126,1,14010315,21:00-23:59,39.00,126,126,39.00
0,6,1,14010316,18:00-20:59,1.00,6,6,1.00


In [51]:
# لیست گروه‌ها
groups = df_grouped[["order_date", "variety_id", "grade"]].drop_duplicates()

# ساخت همه بازه‌ها برای هر گروه
full_index = groups.merge(
    pd.DataFrame({"order_time": time_order}),
    how="cross"
)

# merge با دیتا اصلی
df_full = full_index.merge(
    df_grouped,
    on=["order_date", "variety_id", "grade", "order_time"],
    how="left"
)

df_full.head()

,order_date,variety_id,grade,order_time,amount,source_id,descendants,inventory
0,14010315,56,1,06:00-08:59,NaN,NaN,NaN,NaN
1,14010315,56,1,09:00-11:59,NaN,NaN,NaN,NaN
2,14010315,56,1,12:00-14:59,NaN,NaN,NaN,NaN
3,14010315,56,1,15:00-17:59,NaN,NaN,NaN,NaN
4,14010315,56,1,18:00-20:59,1.51,56,56,1.51


In [52]:
# لیست گروه‌ها
groups = df_grouped[["order_date", "variety_id", "grade"]].drop_duplicates()

# ساخت همه بازه‌ها برای هر گروه
full_index = groups.merge(
    pd.DataFrame({"order_time": time_order}),
    how="cross"
)

# merge با دیتا اصلی
df_full = full_index.merge(
    df_grouped,
    on=["order_date", "variety_id", "grade", "order_time"],
    how="left"
)

# پر کردن source_id و descendants برای هر گروه
keys = ["order_date", "variety_id", "grade"]

df_full["source_id"] = df_full.groupby(keys)["source_id"].transform("first")
df_full["descendants"] = df_full.groupby(keys)["descendants"].transform("first")

df_full.head()

,order_date,variety_id,grade,order_time,amount,source_id,descendants,inventory
0,14010315,56,1,06:00-08:59,NaN,56,56,NaN
1,14010315,56,1,09:00-11:59,NaN,56,56,NaN
2,14010315,56,1,12:00-14:59,NaN,56,56,NaN
3,14010315,56,1,15:00-17:59,NaN,56,56,NaN
4,14010315,56,1,18:00-20:59,1.51,56,56,1.51


In [53]:
# ترتیب زمانی
df_full["order_time"] = pd.Categorical(
    df_full["order_time"],
    categories=time_order,
    ordered=True
)

keys = ["order_date", "variety_id", "grade"]

# NaN -> 0
df_full["amount_filled"] = df_full["amount"].fillna(0)

# مرتب‌سازی برای تضمین ترتیب درست
df_full = df_full.sort_values(keys + ["order_time"])

# inventory کاهشی (amount این بازه در inventory این بازه لحاظ نشود)
df_full["inventory"] = (
    df_full
    .groupby(keys)["amount_filled"]
    .transform(lambda x: x.sum() - x.cumsum().shift(fill_value=0))
)

In [54]:
df_full.head()

,order_date,variety_id,grade,order_time,amount,source_id,descendants,inventory,amount_filled
0,14010315,56,1,06:00-08:59,NaN,56,56,1.51,0.00
1,14010315,56,1,09:00-11:59,NaN,56,56,1.51,0.00
2,14010315,56,1,12:00-14:59,NaN,56,56,1.51,0.00
3,14010315,56,1,15:00-17:59,NaN,56,56,1.51,0.00
4,14010315,56,1,18:00-20:59,1.51,56,56,1.51,1.51


In [55]:
df_full["total_inventory"] = (
    df_full
    .groupby(
        ["order_date", "order_time", "variety_id"],
        observed=False
    )["inventory"]
    .transform("sum")
)

In [56]:
df_full.head()

,order_date,variety_id,grade,order_time,amount,source_id,descendants,inventory,amount_filled,total_inventory
0,14010315,56,1,06:00-08:59,NaN,56,56,1.51,0.00,1.51
1,14010315,56,1,09:00-11:59,NaN,56,56,1.51,0.00,1.51
2,14010315,56,1,12:00-14:59,NaN,56,56,1.51,0.00,1.51
3,14010315,56,1,15:00-17:59,NaN,56,56,1.51,0.00,1.51
4,14010315,56,1,18:00-20:59,1.51,56,56,1.51,1.51,1.51


In [57]:
#basket => 5 
#out =>6
#process => 7

df_full['grade'] = df_full['grade'].replace({
    'basket': 5,
    'out': 6,
    'process': 7
})

df_full = df_full.dropna(subset=['source_id'])

df_full['month'] = df_full['order_date'].astype(str).str[4:6].astype(int)
df_full['season'] = ((df_full['month'] - 1) // 3 + 1).astype(int)

In [58]:
# ذخیره فایل جدید
df_full.to_csv(
    "Data/result_for_model.csv",
    index=False,
    encoding="utf-8-sig"
)